In [ ]:
#------------------------------------------------------------------------------------------------------#
#
# Code:      CC01_CCAR_B03_model_scoring_01.ipynb
#
# Objective: Step B03: Use mortgage data of 2016-04 to score new PD model
#
#            Jingru Chen
#            2026-03-22
#
#----------------------------------------------------------------------------------------------------#

# Step 1: Upload libraries

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

run_date="2026-03-22"
my_vintage = 201604

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM EDT

2026-03-22 15:45:18 EDT
2026-03-22 03:45:18 PM EDT


In [ ]:
myout= "/content/sample_data"

In [ ]:
pwd

'/content'

In [ ]:
cd /content/sample_data/

/content/sample_data


In [ ]:
ls -ltr

total 84036
-rwxr-xr-x 1 root root      962 Jan  1  2000 README.md*
-rwxr-xr-x 1 root root     1697 Jan  1  2000 anscombe.json*
-rw-r--r-- 1 root root  1706430 Mar 17 17:58 california_housing_train.csv
-rw-r--r-- 1 root root   301141 Mar 17 17:58 california_housing_test.csv
-rw-r--r-- 1 root root 36523880 Mar 17 17:58 mnist_train_small.csv
-rw-r--r-- 1 root root 18289443 Mar 17 17:58 mnist_test.csv
-rw-r--r-- 1 root root     2209 Mar 22 17:45 ccar_pd_model_2026-03-22.pkl
-rw-r--r-- 1 root root 29210174 Mar 22 17:45 CCAR_Mortgage_data_for_model_DEV_20260319_01.csv


In [ ]:
x_list=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate', 'loan_term_months',
        'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24']

x_list_v2=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
        'delta_Unemployment1',
       'delta_Unemployment3',
       'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12',
       'delta_Unemployment24' ]

y_list= ['flag_default']

pd_model = 'ccar_pd_model_2026-03-22.pkl'

# Step 2: Upload new PD file via pipeline

In [ ]:
# Load the model from your disk/storage
pd_model = joblib.load( pd_model )

# Verify it's ready
print(f"Model type: {type(pd_model)}")

Model type: <class 'sklearn.pipeline.Pipeline'>


# Step 3: Upload Mortgage data and select the vintage of 2016-04

In [ ]:
df_mortgage= pd.read_csv( myout + "/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv" )
df_mortgage= df_mortgage.drop( columns= ['Unnamed: 0'] )

df_mortgage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67957 entries, 0 to 67956
Data columns (total 44 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   loan_id                           67957 non-null  int64  
 1   origination_date                  67957 non-null  object 
 2   report_date                       67957 non-null  object 
 3   num_payments                      67957 non-null  int64  
 4   original_balance                  67957 non-null  float64
 5   current_balance_bk                67957 non-null  float64
 6   EAD                               67957 non-null  float64
 7   credit_score_orig                 67957 non-null  float64
 8   loan_to_value_orig                67957 non-null  float64
 9   interest_rate                     67957 non-null  float64
 10  product_type                      67957 non-null  object 
 11  ever_defaulted                    67957 non-null  bool   
 12  defa

In [ ]:
df_mortgage.columns

Index(['loan_id', 'origination_date', 'report_date', 'num_payments',
       'original_balance', 'current_balance_bk', 'EAD', 'credit_score_orig',
       'loan_to_value_orig', 'interest_rate', 'product_type', 'ever_defaulted',
       'default_date', 'PD', 'LGD', 'projected_loss', 'loan_term_months',
       'unemployment', 'gdp_growth_qoq', 'hpi_change', 'bbb_spread',
       'months_elapsed', 'current_balance', 'yrmo', 'report_yrmo',
       'default_yrmo', 'flag_default', 'flag_removal', 'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24',
       'flag_merge'],
      dtype='object')

In [ ]:
df_mortgage_vintage= df_mortgage.loc[df_mortgage.report_yrmo == my_vintage].reset_index(drop=True)

df_mortgage_vintage.report_yrmo.value_counts()

,count
report_yrmo,
201604,1179


In [ ]:
pd.crosstab( index= df_mortgage_vintage['report_yrmo'], columns= df_mortgage_vintage['flag_default'],
            margins=True)

flag_default,0,1,All
report_yrmo,,,
201604,1157,22,1179
All,1157,22,1179


In [ ]:
df_mortgage_vintage.shape

(1179, 44)

In [ ]:
df_mortgage_vintage

,loan_id,origination_date,report_date,num_payments,original_balance,current_balance_bk,EAD,credit_score_orig,loan_to_value_orig,interest_rate,...,delta_Unemployment6,delta_Mortgage_rate6,delta_House_Price_Index__Level6,delta_Unemployment12,delta_Mortgage_rate12,delta_House_Price_Index__Level12,delta_Unemployment24,delta_Mortgage_rate24,delta_House_Price_Index__Level24,flag_merge
0,2,2013-04-30,2016-04-30,36,42721.25,24345.71,24345.71,699.0,57.754704,6.440,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both
1,3,2011-05-31,2016-04-30,59,121969.01,74201.76,74201.76,582.0,57.247045,4.948,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both
2,5,2012-07-31,2016-04-30,45,37559.02,27160.26,27160.26,647.0,97.138985,3.627,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both
3,9,2013-10-31,2016-04-30,30,18437.63,13610.61,13610.61,762.0,104.587226,6.475,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both
4,12,2009-11-30,2016-04-30,77,5417.07,2369.79,2369.79,736.0,78.308803,3.533,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1174,4986,2010-11-30,2016-04-30,65,74338.75,34906.79,34906.79,681.0,77.709421,8.916,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both
1175,4987,2012-09-30,2016-04-30,43,291398.08,212997.64,212997.64,708.0,89.422514,3.536,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both
1176,4991,2010-12-31,2016-04-30,64,17879.11,12249.50,12249.50,784.0,86.870733,5.827,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both
1177,4998,2009-06-30,2016-04-30,82,432019.08,318693.10,318693.10,713.0,82.706945,5.444,...,-0.044555,0.07726,0.010883,-0.125911,0.112504,0.0335,-0.265614,-0.023655,0.081832,both


In [ ]:
X = df_mortgage_vintage[ x_list_v2 ]

y = df_mortgage_vintage['flag_default']

print( "---------------- Type of X is ---: ", type(X) )
print( "---------------- Type of y is ---: ", type(y) )

---------------- Type of X is ---:  <class 'pandas.core.frame.DataFrame'>
---------------- Type of y is ---:  <class 'pandas.core.series.Series'>


# Step 4: Run PD model on new vintage data

In [ ]:
# 1. Generate Predictions
# Get probabilities instead of hard predictions
y_proba = pd_model.predict_proba(X)[:, 1]

# Set a custom threshold based on your portfolio's average default rate
custom_threshold = 0.6
y_pred_new = (y_proba >= custom_threshold).astype(int)


# 2. Calculate Individual Metrics
accuracy = accuracy_score( y, y_pred_new )
precision = precision_score( y, y_pred_new )
recall = recall_score(y, y_pred_new )
f1 = f1_score( y, y_pred_new )

# 3. Print the Comprehensive Classification Report
print("--- Classification Report ---")
print(classification_report(y, y_pred_new ))

# 4. Display the Confusion Matrix
print("--- Confusion Matrix ---")
print(confusion_matrix(y, y_pred_new ))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1157
           1       0.05      0.05      0.05        22

    accuracy                           0.97      1179
   macro avg       0.52      0.51      0.52      1179
weighted avg       0.96      0.97      0.97      1179

--- Confusion Matrix ---
[[1139   18]
 [  21    1]]


In [ ]:
print( "-------4-01: Shape of df_mortgage_vintag ----:", df_mortgage_vintage.shape )

print( "-------4-02: Type of y_pred_new ----:", type( y_pred_new ) )
y_pred_new_pd = pd.DataFrame( y_pred_new, columns=["y_pred_new"] )
print( "-------4-03: Type of y_pred_new_pd ----:", type( y_pred_new_pd ) )

print( "-------4-04: Column List of y_pred_new_pd ----:", y_pred_new_pd.columns )

df_mortgage_vintage_v1 = pd.concat( [df_mortgage_vintage, y_pred_new_pd], axis=1)

print( "-------4-05: Column List of df_mortgage_vintage_v1 ----:", df_mortgage_vintage_v1.shape )

df_mortgage_vintage_v1.columns

-------4-01: Shape of df_mortgage_vintag ----: (1179, 44)
-------4-02: Type of y_pred_new ----: <class 'numpy.ndarray'>
-------4-03: Type of y_pred_new_pd ----: <class 'pandas.core.frame.DataFrame'>
-------4-04: Column List of y_pred_new_pd ----: Index(['y_pred_new'], dtype='object')
-------4-05: Column List of df_mortgage_vintage_v1 ----: (1179, 45)


Index(['loan_id', 'origination_date', 'report_date', 'num_payments',
       'original_balance', 'current_balance_bk', 'EAD', 'credit_score_orig',
       'loan_to_value_orig', 'interest_rate', 'product_type', 'ever_defaulted',
       'default_date', 'PD', 'LGD', 'projected_loss', 'loan_term_months',
       'unemployment', 'gdp_growth_qoq', 'hpi_change', 'bbb_spread',
       'months_elapsed', 'current_balance', 'yrmo', 'report_yrmo',
       'default_yrmo', 'flag_default', 'flag_removal', 'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24',
       'flag_merge', 'y_pred_new'],
      dt

In [ ]:
from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-03-22 15:45:18.422417-04:00
Finished: 2026-03-22 15:45:20.682219-04:00

Duration: 0:00:02.259802
Duration: 2.260 seconds
